In [ ]:
!pip install -q x-transformers

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import os
import sys
import subprocess
import hashlib
import gc
from datetime import datetime
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from transformers import RobertaTokenizerFast, get_cosine_schedule_with_warmup, DataCollatorForLanguageModeling
from datasets import load_dataset
from x_transformers import Encoder

# ==========================================
# 1. CONFIGURATION
# ==========================================
# YOUR REPO ID (Created in previous step)
HF_ID = "prism-lab/wikitext-103-prism-32k-seq4k"

# Hyperparameters
VOCAB_SIZE = 32768
SEQ_LEN = 4096
BATCH_SIZE = 8
EPOCHS = 40
LR = 1e-3
D_MODEL = 512
DEPTH = 6
DROPOUT = 0.1
RESUME_PATH = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

# ==========================================
# 2. DATA PIPELINE (The "Pro" Way)
# ==========================================
def prepare_data_from_hub():
    print(f"⬇️ Pulling Pre-Tokenized Data from {HF_ID}...")

    # 1. Load Tokenizer (Instant)
    # This pulls the exact tokenizer you uploaded
    tokenizer = RobertaTokenizerFast.from_pretrained(HF_ID)

    # 2. Load Dataset (Instant)
    # This pulls the already chunked/tokenized data
    dataset = load_dataset(HF_ID)

    print(f"✅ Loaded {len(dataset['train'])} training chunks.")

    # 3. Collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    return dataset, data_collator
# ==========================================
# 3. PRISM ARCHITECTURE (Complex-Valued)
# ==========================================

class ComplexDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p
    def forward(self, z):
        if not self.training or self.p == 0.0: return z
        mask = torch.ones_like(z.real)
        mask = F.dropout(mask, self.p, self.training, inplace=False)
        return z * mask

class RobustPhaseNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        mag = torch.abs(x)
        rms = torch.sqrt(torch.mean(mag**2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale

class ModReLU(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.b = nn.Parameter(torch.zeros(features))
    def forward(self, z):
        mag = torch.abs(z)
        new_mag = F.relu(mag + self.b)
        phase = z / (mag + 1e-6)
        return new_mag * phase

class ComplexToRealBridge(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x_complex):
        cat = torch.cat([x_complex.real, x_complex.imag], dim=-1)
        return self.norm(self.proj(cat))

# ==========================================
# 4. DYNAMIC RoSE (Mamba-3 Engine)
# ==========================================
class DynamicRoSE(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, max_period=10000.0):
        super().__init__()
        self.embedding_dim = embedding_dim

        # 1. Master Real Embedding (The "Particle")
        self.raw_embedding = nn.Embedding(num_embeddings, embedding_dim)

        # 2. Complex Adapter (The "Wave" Magnitude/Initial Phase)
        self.adapter = nn.Linear(embedding_dim, embedding_dim * 2)

        # 3. Static Frequencies (Positional)
        freqs = torch.exp(torch.arange(0, embedding_dim, dtype=torch.float32) * -(math.log(max_period) / embedding_dim))
        self.register_buffer('freqs', freqs)

        self.rotation_predictor = nn.Linear(embedding_dim, embedding_dim * 2)

    def forward(self, input_ids):
        # A. Raw Particle
        real_base = self.raw_embedding(input_ids)
        B, L, D = real_base.shape

        # B. Complex Wave Content
        complex_params = self.adapter(real_base)
        z_t = torch.complex(complex_params[..., :D], complex_params[..., D:])

        rot_raw = self.rotation_predictor(real_base)
        rot_x, rot_y = rot_raw.chunk(2, dim=-1)

        rot_mag = torch.sqrt(rot_x**2 + rot_y**2 + 1e-6)
        dynamic_rot = torch.complex(rot_x / rot_mag, rot_y / rot_mag)

        # D. Static Positional Rotation
        pos = torch.arange(L, device=input_ids.device).float()
        static_angles = torch.outer(pos, self.freqs) # [L, D]
        static_rot = torch.polar(torch.ones_like(static_angles), static_angles) # [L, D]

        z_final = z_t * static_rot.unsqueeze(0) * dynamic_rot

        return z_final, real_base

# ==========================================
# 5. HYENA FILTER
# ==========================================
class HyenaNeuralFilter(nn.Module):
    def __init__(self, d_model, max_len=1024, hidden_dim=64):
        super().__init__()
        self.d_model = d_model
        freqs = torch.exp(torch.arange(0, hidden_dim, 2, dtype=torch.float32) * -(math.log(10000.0) / hidden_dim))
        self.register_buffer("freqs", freqs)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, d_model * 2)
        )
    def forward(self, L, device):
        t = torch.linspace(0, 1, steps=L, device=device).unsqueeze(-1)
        emb = torch.cat([torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        out = self.mlp(emb).view(L, self.d_model, 2)
        return torch.complex(out[..., 0], out[..., 1])

# ==========================================
# 6. GATED HARMONIC CONVOLUTION (Lean)
# ==========================================
class GatedHarmonicConvolution(nn.Module):
    def __init__(self, d_model, max_len=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.filter_len = max_len
        self.neural_filter = HyenaNeuralFilter(d_model, max_len=max_len)
        self.gate_proj = nn.Linear(d_model * 2, d_model * 2)
        self.mix_real = nn.Linear(d_model, d_model)
        self.mix_imag = nn.Linear(d_model, d_model)
        self.out_real = nn.Linear(d_model, d_model)
        self.out_imag = nn.Linear(d_model, d_model)
        self.activation = ModReLU(d_model)
        self.norm = RobustPhaseNorm(d_model)
        self.dropout = ComplexDropout(dropout)

    def forward(self, x, src_mask=None):
        residual = x
        x_norm = self.norm(x)
        if src_mask is not None:
             x_norm = x_norm.masked_fill(src_mask.unsqueeze(-1), 0.0)

        # 1. Global Beam (FFT + Hyena)
        B, L, D = x_norm.shape
        eff_L = min(L, self.filter_len)
        x_freq = torch.fft.fft(x_norm, n=eff_L, dim=1, norm='ortho')
        h = self.neural_filter(eff_L, x.device).unsqueeze(0)
        x_filtered = x_freq * h
        x_time = torch.fft.ifft(x_filtered, n=eff_L, dim=1, norm='ortho')
        if L > eff_L: x_time = F.pad(x_time, (0,0,0,L-eff_L))
        else: x_time = x_time[:, :L, :]

        # 2. Gating
        gates = torch.sigmoid(self.gate_proj(torch.cat([x_norm.real, x_norm.imag], dim=-1)))
        g_r, g_i = gates.chunk(2, dim=-1)
        x_gated = torch.complex(x_time.real * g_r, x_time.imag * g_i)

        # 3. Mixing & Out
        mr, mi = self.mix_real, self.mix_imag
        x_mixed = torch.complex(mr(x_gated.real) - mi(x_gated.imag), mr(x_gated.imag) + mi(x_gated.real))
        x_act = self.activation(x_mixed)
        or_, oi = self.out_real, self.out_imag
        out = torch.complex(or_(x_act.real) - oi(x_act.imag), or_(x_act.imag) + oi(x_act.real))
        return self.dropout(out) + residual

# ==========================================
# 7. MODEL WRAPPERS
# ==========================================
class PRISMEncoder(nn.Module):
    def __init__(self, num_layers, d_model, max_len, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            GatedHarmonicConvolution(d_model, max_len, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = RobustPhaseNorm(d_model)
    def forward(self, x, src_mask=None):
        for layer in self.layers:
            if self.training: x = torch.utils.checkpoint.checkpoint(layer, x, src_mask, use_reentrant=False)
            else: x = layer(x, src_mask)
        return self.final_norm(x)

class PRISM_WikiText_Model(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, prism_depth=5, trans_depth=1, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # 1. PRISM Core (The Optical/Passive Part)
        self.rose = DynamicRoSE(vocab_size, d_model)
        self.prism_encoder = PRISMEncoder(prism_depth, d_model, max_len=max_len, dropout=dropout)
        self.bridge = ComplexToRealBridge(d_model)
        self.periscope_proj = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model), nn.GELU())

        # 2. Refiner (The Digital/Active Part)
        # 🔄 SWAPPED: Replaced Standard Transformer with RoPE-Enabled Encoder
        if trans_depth > 0:
            self.refiner = Encoder(
                dim=d_model,
                depth=trans_depth,
                heads=8,
                rotary_pos_emb=True,
                attn_flash=True,
                attn_dropout=dropout,
                ff_dropout=dropout,

            )
        else:
            self.refiner = None

        # 3. Output
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.lm_head.weight = self.rose.raw_embedding.weight

    def forward(self, input_ids):
        # A. Wave Physics
        wave_src, particle_src = self.rose(input_ids)
        wave_out = self.prism_encoder(wave_src)
        wave_real = self.bridge(wave_out)

        # B. Interface
        mixed_memory = self.periscope_proj(torch.cat([wave_real, particle_src], dim=-1))

        # C. Digital Refinement (Now with RoPE)
        if self.refiner:
            out = self.refiner(mixed_memory)
        else:
            out = mixed_memory

        return self.lm_head(out)

In [ ]:
# ==========================================
# 4. LOGGING UTILITIES
# ==========================================
def generate_run_id():
    raw = datetime.now().strftime("%Y%m%d%H%M%S%f")
    return hashlib.md5(raw.encode()).hexdigest()[:8]

def log_environment(save_dir, run_id, config):
    log_path = os.path.join(save_dir, f"env_metadata_{run_id}.txt")
    with open(log_path, "w") as f:
        f.write(f"PRISM EXPERIMENT METADATA | Run ID: {run_id}\n{'='*50}\n")
        for k, v in config.items(): f.write(f"{k}: {v}\n")
    print(f"📝 Environment Snapshot saved to: {log_path}")

def log_metrics(save_dir, run_id, epoch, train_loss, val_loss, ppl):
    log_path = os.path.join(save_dir, f"metrics_log_{run_id}.csv")
    if not os.path.exists(log_path):
        with open(log_path, "w") as f: f.write("Timestamp,Epoch,Train_Loss,Val_Loss,Perplexity\n")
    with open(log_path, "a") as f:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"{ts},{epoch},{train_loss:.6f},{val_loss:.6f},{ppl:.6f}\n")


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_loss, config):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_loss,
        'config': config
    }, path)


def run_wikitext_training(experiment_name="PRISM2_WT103_40epochs"):
    from google.colab import drive
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')

    # --- SETUP DIRS ---
    if RESUME_PATH and os.path.exists(RESUME_PATH):
        print(f"🔄 RESUMING FROM: {RESUME_PATH}")
        checkpoint = torch.load(RESUME_PATH, map_location=DEVICE)
        SAVE_DIR = os.path.dirname(RESUME_PATH)
        run_id = checkpoint.get('config', {}).get('run_id', 'resumed')
    else:
        run_id = hashlib.md5(datetime.now().strftime("%Y%m%d%H%M%S%f").encode()).hexdigest()[:8]
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        folder_name = f"{experiment_name}_{timestamp}_{run_id}"
        SAVE_DIR = os.path.join("/content/drive/My Drive/PRISM_Experiments", folder_name)
        os.makedirs(SAVE_DIR, exist_ok=True)
        print(f"💾 Checkpoints: {SAVE_DIR}")

    writer = SummaryWriter(log_dir=SAVE_DIR)
    GRAD_ACCUM = 4

    lm_datasets, data_collator = prepare_data_from_hub()

    # WORKERS=2 (Safe for Colab)
    train_loader = DataLoader(
        lm_datasets["train"], batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=data_collator, num_workers=2, pin_memory=True,
        prefetch_factor=2, persistent_workers=True
    )
    valid_loader = DataLoader(
        lm_datasets["validation"], batch_size=BATCH_SIZE,
        collate_fn=data_collator, num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        lm_datasets["test"], batch_size=BATCH_SIZE,
        collate_fn=data_collator, num_workers=2, pin_memory=True
    )

    print("\n⚡ INITIALIZING MODEL...")
    model = PRISM_WikiText_Model(
        vocab_size=VOCAB_SIZE, d_model=D_MODEL, max_len=SEQ_LEN,
        prism_depth=DEPTH-1, trans_depth=1, dropout=DROPOUT
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0)
    total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss()

    start_epoch = 0
    best_val_loss = float('inf')

    if RESUME_PATH and os.path.exists(RESUME_PATH):
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        del checkpoint
        torch.cuda.empty_cache()
    else:
        def init_weights_PRISM(m):
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=D_MODEL**-0.5)
        model.apply(init_weights_PRISM)
        nn.init.normal_(model.rose.rotation_predictor.weight, std=0.01)
        with torch.no_grad():
            model.rose.rotation_predictor.bias[:D_MODEL].fill_(1.0)
            model.rose.rotation_predictor.bias[D_MODEL:].fill_(0.0)

    print(f"\n🚀 STARTING (Ep {start_epoch+1} to {EPOCHS})")
    global_step = (len(train_loader) // GRAD_ACCUM) * start_epoch

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS}")

        for step, batch in enumerate(pbar):
            x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)

            # Forward
            loss = criterion(model(x).view(-1, VOCAB_SIZE), y.view(-1)) / GRAD_ACCUM
            loss.backward()

            # Step (Every 4 batches)
            if (step + 1) % GRAD_ACCUM == 0:
                # 1. Calc Norm
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                # 2. Step
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                # 3. UPDATE BAR WITH GNORM
                actual_loss = loss.item() * GRAD_ACCUM
                writer.add_scalar('Train/Loss', actual_loss, global_step)

                # <--- FIXED LINE HERE:
                pbar.set_postfix({
                    'loss': f"{actual_loss:.4f}",
                    'gnorm': f"{grad_norm.item():.2f}"
                })

        # VALIDATION
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in valid_loader:
                x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)
                val_loss += criterion(model(x).view(-1, VOCAB_SIZE), y.view(-1)).item()

        avg_val_loss = val_loss / len(valid_loader)
        ppl = math.exp(avg_val_loss) if avg_val_loss < 100 else float('inf')

        print(f"✨ Epoch {epoch+1} | Val Loss: {avg_val_loss:.4f} | PPL: {ppl:.2f}")
        writer.add_scalar('Val/PPL', ppl, epoch+1)

        config_dump = {"epoch": epoch, "run_id": run_id}
        save_checkpoint(os.path.join(SAVE_DIR, "last.pt"), model, optimizer, scheduler, epoch, best_val_loss, config_dump)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best.pt"))
            print("   🏆 New Best Model Saved!")

    # TEST
    best_path = os.path.join(SAVE_DIR, "best.pt")
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path))
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Testing"):
                x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)
                test_loss += criterion(model(x).view(-1, VOCAB_SIZE), y.view(-1)).item()
        print(f"🏆 FINAL PPL: {math.exp(test_loss/len(test_loader)):.2f}")

    writer.close()
    return model

In [ ]:
def analyze_prism_params(model):
    print("="*80)
    print(f"📊 LEAN PRISM-2 PARAMETER ANALYSIS")
    print("="*80)
    total_params = sum(p.numel() for p in model.parameters())
    # Embeddings (Shared)
    vocab_params = model.rose.raw_embedding.weight.numel()
    # Wave Engine
    enc_params = sum(p.numel() for p in model.prism_encoder.parameters())
    # Transformer Refiner
    ref_params = sum(p.numel() for p in model.refiner.parameters()) if model.refiner else 0
    # Other
    other_params = total_params - vocab_params - enc_params - ref_params

    print(f"{'Shared Embeddings (Particle)':<35} | {vocab_params:<15,} | {vocab_params/total_params:.1%} | Tied")
    print(f"{'PRISM Optical Engine':<35} | {enc_params:<15,} | {enc_params/total_params:.1%} | 5 Layers")
    print(f"{'Digital Refiner':<35} | {ref_params:<15,} | {ref_params/total_params:.1%} | 1 Layer")
    print("="*80)
    print(f"{'TOTAL PARAMETERS':<35} | {total_params:<15,} | 100.0%")
    print("="*80)


if __name__ == "__main__":

    print("🏗️ INSTANTIATING MODEL FOR INSPECTION...")
    # Initialize a temporary model just for counting
    dummy_model = PRISM_WikiText_Model(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        max_len=SEQ_LEN,
        prism_depth=DEPTH-1,
        trans_depth=1,
        dropout=DROPOUT
    )

    # 2. Run Analysis
    analyze_prism_params(dummy_model)

    # 3. Clean up to free RAM for actual training
    del dummy_model
    gc.collect()
    torch.cuda.empty_cache()

    # 4. Ask for confirmation (Optional, or just proceed)
    print("\n✅ Analysis Complete. Starting Training Routine in 5 seconds...")
    import time
    time.sleep(5)

    # 5. Start Training
    trained_prism = run_wikitext_training()

    # 6. Final Analysis (Post-training check)
    analyze_prism_params(trained_prism)

    # 7. Kill Runtime (Colab specific)
    from google.colab import runtime
    runtime.unassign()

In [ ]:
from google.colab import runtime
runtime.unassign()